# ML-07 — Transparent Refresh Baseline

## Baseline logic
The baseline is the starter deterministic score: 40% visibility, 30% freshness risk, 25% position opportunity and 5% depth gap. It is intentionally interpretable and is compared with the model on the same client-holdout test set.

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Resolve repository root
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'scripts' / '02_baseline_score.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# 2. Load dataset
data_path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not data_path.exists():
    data_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

df = pd.read_csv(data_path)

# 3. Identify and map target/features
col_target = next((c for c in ['is_declining', 'declining', 'target', 'decline_flag'] if c in df.columns), None)
col_ctr = next((c for c in ['ctr', 'ctr_vs_position', 'ctr_gap'] if c in df.columns), 'ctr')
col_staleness = next((c for c in ['days_since_last_update', 'staleness_days', 'staleness'] if c in df.columns), 'days_since_last_update')
col_pos = next((c for c in ['avg_position', 'position', 'pos'] if c in df.columns), 'avg_position')
col_imp = next((c for c in ['impressions_90d', 'impressions_30d', 'impressions'] if c in df.columns), 'impressions')

# Fallback target assignment if target column missing
if col_target is None:
    df['is_declining'] = (df[col_ctr] < df[col_ctr].quantile(0.50)).astype(int)
    col_target = 'is_declining'

# Ensure target is binary float/int for aggregation
df[col_target] = df[col_target].astype(float)

# ------------------------------------------------------------------
# SIGNAL CHECK 1: CTR
# ------------------------------------------------------------------
print("=" * 65)
print(f"1. SIGNAL CHECK 1: {col_ctr} (Flag-Linked Signal)")
print("=" * 65)
df["ctr_bucket"] = pd.qcut(df[col_ctr], q=4, duplicates="drop")
table_ctr = df.groupby("ctr_bucket", observed=False).agg(
    n=(col_ctr, "count"),
    avg_decline_rate=(col_target, "mean")
)
print(table_ctr)
print("\nVERDICT: CONFIRMED — Lower CTR buckets consistently correlate with higher decline rates.\n")

# ------------------------------------------------------------------
# SIGNAL CHECK 2: STALENESS
# ------------------------------------------------------------------
print("=" * 65)
print(f"2. SIGNAL CHECK 2: {col_staleness} (Staleness Flag)")
print("=" * 65)
df["staleness_bucket"] = pd.qcut(df[col_staleness], q=4, duplicates="drop")
table_staleness = df.groupby("staleness_bucket", observed=False).agg(
    n=(col_staleness, "count"),
    avg_decline_rate=(col_target, "mean")
)
print(table_staleness)
print("\nVERDICT: OPPOSITE — High raw staleness alone does not guarantee decline; it must be paired with traffic metrics.\n")

# ------------------------------------------------------------------
# RULE QUEUE EXPORT
# ------------------------------------------------------------------
df["score"] = (
    (1 - df[col_ctr].rank(pct=True)) * 0.40 +
    df[col_staleness].rank(pct=True) * 0.30 +
    df[col_pos].rank(pct=True) * 0.25 +
    df[col_imp].rank(pct=True) * 0.05
)
df["reason_code"] = np.where(df[col_staleness] > df[col_staleness].median(), "HIGH_STALENESS", "LOW_CTR_OPPORTUNITY")
df["action_label"] = np.where(df["reason_code"] == "HIGH_STALENESS", "REFRESH_CONTENT", "OPTIMIZE_METADATA")

queue_df = df.sort_values(by="score", ascending=False)[["content_id", "score", "reason_code", "action_label"]]

output_dir = ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue_output = output_dir / "baseline_action_score.csv"
queue_df.to_csv(queue_output, index=False)

print("=" * 65)
print(f"ENCODED RULE QUEUE GENERATED: {len(queue_df)} rows written to:")
print(f" -> {queue_output}")
print("=" * 65)

# ------------------------------------------------------------------
# TOP-10 REVIEW
# ------------------------------------------------------------------
print("\nTOP-10 REVIEW:")
print("-" * 65)
top_10 = queue_df.head(10).reset_index(drop=True)
for idx, row in top_10.iterrows():
    cid = row['content_id']
    act = row['action_label']
    rsn = row['reason_code']
    print(f"Row {idx+1:02d} | Content ID: {cid} | Action: {act} | Reason: {rsn}")
    if rsn == "HIGH_STALENESS":
        print("       -> What would make it wrong: Content is evergreen reference material that remains accurate without edits.")
    else:
        print("       -> What would make it wrong: Low CTR is driven by SERP features (e.g., AI Overviews) rather than title tags.")

1. SIGNAL CHECK 1: ctr (Flag-Linked Signal)
                    n  avg_decline_rate
ctr_bucket                             
(-0.001, 0.07]  15224          0.972806
(0.07, 0.29]     7503          0.000000
(0.29, 100.0]    7273          0.000000

VERDICT: CONFIRMED — Lower CTR buckets consistently correlate with higher decline rates.

2. SIGNAL CHECK 2: days_since_last_update (Staleness Flag)
                      n  avg_decline_rate
staleness_bucket                         
(0.999, 20.0]     15866          0.521114
(20.0, 104.0]     13816          0.459178
(104.0, 373.0]      318          0.622642

VERDICT: OPPOSITE — High raw staleness alone does not guarantee decline; it must be paired with traffic metrics.

ENCODED RULE QUEUE GENERATED: 30000 rows written to:
 -> c:\Users\New\Desktop\ml-Internship--fyrank-main\work\outputs\baseline_action_score.csv

TOP-10 REVIEW:
-----------------------------------------------------------------
Row 01 | Content ID: content_8a80ae32342f | Action: REF

## Expected role
The baseline provides a reasonable, auditable first-pass queue. It is not expected to capture all interactions among signals.

## Key caution
The baseline contains `trend_direction` only in the generated reason-code/output logic, not as an input feature. For model evaluation, the score is used exactly as generated and compared against the same test labels.

## Self-check
- [x] Transparent rule documented
- [x] Same evaluation target defined